Purpose: Set up data inputs for *Y. gloriosa* single-replicate (per genotype-treatment-ZT) maSigPro run.<br>
Author: Anna Pardo<br>
Date initiated: Mar. 31, 2026

In [8]:
import pandas as pd
import numpy as np
import json

In [2]:
# load counts
counts = pd.read_csv("/home/leviathan22/Yucca_genomics/rna_insilico_genome/counts/Yg_toYgIS_all_md_countsmatrix_over1mil.txt",
                    sep="\t",header="infer")
counts.head()

/tmp/ipykernel_1209/168404881.py:2: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  counts = pd.read_csv("/home/leviathan22/Yucca_genomics/rna_insilico_genome/counts/Yg_toYgIS_all_md_countsmatrix_over1mil.txt",


,sample_name,genotype,time,treat,ZT,species,Yucal.01G000100.v2.1,Yucal.01G000200.v2.1,Yucal.01G000300.v2.1,Yucal.01G000400.v2.1,...,YufilH1095123m.g,YufilH1095125m.g,YufilH1095126m.g,YufilH1095128m.g,YufilH1095131m.g,YufilH1095132m.g,YufilH1095134m.g,YufilH1095146m.g,YufilH1095147m.g,Total_Reads
0,Y1,18,1.0,W,1.0,gloriosa,745,50,0,536,...,42,26,14,196,10,34,173,0,0,12458673
1,Y10,2AB,1.5,W,3.0,gloriosa,176,8,0,93,...,3,57,12,35,0,8,63,0,0,2479905
2,Y100,2AB,6.5,W,23.0,gloriosa,472,26,0,246,...,7,179,41,80,12,17,297,0,3,6344543
3,SRR10848707,55,4.0,D,13.0,gloriosa,523,70,0,232,...,22,404,41,70,142,29,84,0,3,10585018
4,SRR10848820,70,4.0,D,13.0,gloriosa,943,136,0,668,...,81,65,59,215,144,100,298,0,0,21651248


In [3]:
counts["genotype"] = counts["genotype"].astype(str)
counts["genotype"].unique()

array(['18', '2AB', '55', '70', '61', '51', '1AB', '19', '15', 'Eudy',
       'G', '56', '36', '13', '13.0', '18.0', '45.0', '45', '52', '46',
       '43', '37', '53', '48', '51.0', '36.0', '16', '6', '12', '20',
       '50'], dtype=object)

In [4]:
gtmap = {}
for g in counts["genotype"].unique():
    if g.endswith(".0"):
        gtmap[g] = g.rstrip(".0")
    else:
        gtmap[g] = g

In [21]:
counts["genotype"] = counts["genotype"].map(gtmap)
counts["genotype"].unique()

array(['18', '2AB', '55', '70', '61', '51', '1AB', '19', '15', 'Eudy',
       'G', '56', '36', '13', '45', '52', '46', '43', '37', '53', '48',
       '16', '6', '12', '20', '50'], dtype=object)

In [5]:
len(counts["genotype"].unique())

31

In [6]:
def getrepsgt(gt,df=counts):
    gtdf = df[df["genotype"]==gt]
    dfl = []
    for t in gtdf["treat"].unique():
        sdf = gtdf[gtdf["treat"]==t]
        for z in [1.0,5.0,9.0,13.0,17.0,21.0]:
            if z in list(sdf["ZT"]):
                sdf2 = sdf[sdf["ZT"]==z]
                dfl.append(sdf2.sample(n=1,axis=0))
    randreps = pd.concat(dfl)
    return randreps

In [7]:
dfl = []
for g in counts["genotype"].unique():
    dfl.append(getrepsgt(g))
subcounts = pd.concat(dfl)
subcounts.head()

,sample_name,genotype,time,treat,ZT,species,Yucal.01G000100.v2.1,Yucal.01G000200.v2.1,Yucal.01G000300.v2.1,Yucal.01G000400.v2.1,...,YufilH1095123m.g,YufilH1095125m.g,YufilH1095126m.g,YufilH1095128m.g,YufilH1095131m.g,YufilH1095132m.g,YufilH1095134m.g,YufilH1095146m.g,YufilH1095147m.g,Total_Reads
560,Y7,18,1.0,W,1.0,gloriosa,1101,80,1,353,...,39,16,26,153,10,29,133,0,0,9290192
189,Y24,18,2.0,W,5.0,gloriosa,572,46,0,345,...,25,26,9,94,18,24,101,0,0,7242045
299,Y40,18,3.0,W,9.0,gloriosa,674,39,0,310,...,23,27,14,141,75,35,51,0,0,6950412
464,Y60,18,4.0,W,13.0,gloriosa,594,45,0,281,...,24,43,29,151,82,36,189,0,0,8516304
541,Y68,18,5.0,W,17.0,gloriosa,343,39,0,249,...,23,22,229,82,39,16,121,0,0,5961077


In [25]:
len(subcounts.index)

302

In [26]:
len(subcounts.index)==len(subcounts["sample_name"].unique())

True

In [27]:
len(counts.index)

850

In [28]:
len(subcounts["genotype"].unique())

26

In [29]:
# save subcounts
subcounts.to_csv("./counts_Yg_singlereps.txt",sep="\t",header=True,index=False)

## Apr. 9: Set up counts for three genotypes only

In [9]:
# first genotype: the one with the most time-structured genes when run individually through maSigPro with no downsampling (37)
## select two others of different physiological classes
phys = json.load(open("/home/leviathan22/Yucca_genomics/phys_figures/physiological_CAM_categories.json"))

In [30]:
# selected genotypes
gts = ["37","61","43"]

In [37]:
# modified function
def getrepsgt(gt,df=counts):
    gtdf = df[df["genotype"]==gt]
    dfl = []
    for t in gtdf["treat"].unique():
        sdf = gtdf[gtdf["treat"]==t]
        for z in [1.0,5.0,9.0,13.0,17.0,21.0]:
            if z in list(sdf["ZT"]):
                sdf2 = sdf[sdf["ZT"]==z]
                dfl.append(sdf2.sample(n=2,axis=0))
    randreps = pd.concat(dfl)
    return randreps

In [31]:
dflist = []
for i in gts:
    print(i)
    dflist.append(getrepsgt(i))

37
61
43


In [32]:
subct = pd.concat(dflist)

In [33]:
subct.head()

,sample_name,genotype,time,treat,ZT,species,Yucal.01G000100.v2.1,Yucal.01G000200.v2.1,Yucal.01G000300.v2.1,Yucal.01G000400.v2.1,...,YufilH1095123m.g,YufilH1095125m.g,YufilH1095126m.g,YufilH1095128m.g,YufilH1095131m.g,YufilH1095132m.g,YufilH1095134m.g,YufilH1095146m.g,YufilH1095147m.g,Total_Reads
847,Y712,37,1.0,W,1.0,gloriosa,643,103,0,632,...,60,892,76,228,16,39,430,0,0,22957719
570,SRR10663376,37,1.0,W,1.0,gloriosa,741,144,0,718,...,55,898,55,174,19,46,379,0,3,22229464
569,SRR10663377,37,2.0,W,5.0,gloriosa,784,134,0,660,...,80,867,38,306,248,38,268,0,11,26038083
574,SRR10663404,37,2.0,W,5.0,gloriosa,564,135,0,452,...,68,897,27,279,218,43,192,0,8,21682455
566,SRR10663382,37,3.0,W,9.0,gloriosa,572,80,0,483,...,37,1006,84,298,300,57,101,0,35,21469449


In [34]:
len(subct.index)

72

In [38]:
dfl = []
for i in gts:
    dfl.append(getrepsgt(i,subct))
rep1 = pd.concat(dfl)

In [39]:
len(rep1.index)

36

In [40]:
rep1.to_csv("./formsp_1rep_37_61_43.txt",sep="\t",header=True,index=False)
subct.to_csv("./formsp_2rep_37_61_43.txt",sep="\t",header=True,index=False)